# Multi-Feature Stock & Crypto LSTM Prediction (Google Colab Edition)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

This notebook provides a **complete, self-contained, GPU-accelerated pipeline** for:
1. **Phase 1 Training**: Ingesting market data (via your custom REST API or public feeds) and engineering technical indicators (SMA, EMA, RSI, MACD, Bollinger Bands, Volatility).
2. **Multi-Feature Benchmarking**: Training and comparing 4 architectures (*Univariate*, *OHLCV*, *Trend/MAs*, and *Full Multi-Feature*) on side-by-side comparative graphs.
3. **Multi-Step Ahead Forecasting**: Projecting N candles ahead (t+1 to t+N) and visualizing error degradation curves.
4. **Phase 2 Live Inference**: Real-time next-candle trajectory prediction on live streaming data.

---
## Step 1: Environment Setup & GPU Acceleration
Let's install dependencies and enable GPU acceleration.

In [ ]:
# Install required packages
!pip install -q yfinance scikit-learn matplotlib pandas numpy torch

import os
import time
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Hardware Acceleration Check
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[*] Active Device: {device.type.upper()}')
if device.type == 'cuda':
    print(f'    GPU Hardware: {torch.cuda.get_device_name(0)}')
else:
    print('    [Tip] Enable GPU in Colab via: Runtime -> Change runtime type -> T4 GPU')


---
## Step 2: PyTorch LSTM Model Architecture
Stacked LSTM with Layer Normalization, Dropout, and dynamic multi-step linear regression head.

In [ ]:
class StockLSTM(nn.Module):
    def __init__(
        self,
        input_size: int,
        hidden_size: int = 64,
        num_layers: int = 2,
        output_size: int = 1,
        dropout: float = 0.2,
        bidirectional: bool = False,
    ):
        super(StockLSTM, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        self.output_size = output_size

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
            bidirectional=bidirectional,
        )

        fc_input_dim = hidden_size * (2 if bidirectional else 1)
        dense_dim = max(64, output_size * 4)

        self.layer_norm = nn.LayerNorm(fc_input_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(fc_input_dim, dense_dim)
        self.relu = nn.ReLU()
        self.fc_out = nn.Linear(dense_dim, output_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        lstm_out, _ = self.lstm(x)
        last_step = lstm_out[:, -1, :]
        normed = self.layer_norm(last_step)
        dropped = self.dropout(normed)
        dense1 = self.relu(self.fc1(dropped))
        return self.fc_out(dense1)

print('[+] StockLSTM model architecture compiled.')


---
## Step 3: Data Pipeline & Technical Indicator Engineering
Computes technical indicators (SMA, EMA, RSI, MACD, Bollinger Bands, Volatility) and creates sliding window sequences without temporal data leakage.

In [ ]:
class StockDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        return self.X[idx], self.y[idx]


def add_technical_indicators(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()
    # 1. Moving Averages
    data['SMA_10'] = data['Close'].rolling(window=10).mean()
    data['SMA_20'] = data['Close'].rolling(window=20).mean()
    data['SMA_50'] = data['Close'].rolling(window=50).mean()
    data['EMA_12'] = data['Close'].ewm(span=12, adjust=False).mean()
    data['EMA_26'] = data['Close'].ewm(span=26, adjust=False).mean()

    # 2. MACD
    data['MACD'] = data['EMA_12'] - data['EMA_26']
    data['MACD_Signal'] = data['MACD'].ewm(span=9, adjust=False).mean()
    data['MACD_Hist'] = data['MACD'] - data['MACD_Signal']

    # 3. RSI 14
    delta = data['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / (loss + 1e-9)
    data['RSI_14'] = 100 - (100 / (1 + rs))

    # 4. Bollinger Bands
    rolling_mean_20 = data['Close'].rolling(window=20).mean()
    rolling_std_20 = data['Close'].rolling(window=20).std()
    data['BB_Upper'] = rolling_mean_20 + (2 * rolling_std_20)
    data['BB_Lower'] = rolling_mean_20 - (2 * rolling_std_20)
    data['BB_Width'] = (data['BB_Upper'] - data['BB_Lower']) / (rolling_mean_20 + 1e-9)

    # 5. Volatility & Return
    data['Daily_Return'] = data['Close'].pct_change()
    data['Volatility_20'] = data['Daily_Return'].rolling(window=20).std()
    return data.dropna()


def create_sliding_sequences(data_matrix: np.ndarray, target_idx: int, seq_length: int = 30, forecast_horizon: int = 1) -> Tuple[np.ndarray, np.ndarray]:
    X, y = [], []
    num_records = len(data_matrix)
    for i in range(num_records - seq_length - forecast_horizon + 1):
        X.append(data_matrix[i : i + seq_length, :])
        y.append(data_matrix[i + seq_length : i + seq_length + forecast_horizon, target_idx])
    return np.array(X), np.array(y)


def prepare_datasets(df: pd.DataFrame, feature_cols: List[str], target_col: str = 'Close', seq_length: int = 30, forecast_horizon: int = 1, train_ratio: float = 0.70, val_ratio: float = 0.15) -> Dict:
    target_idx = feature_cols.index(target_col)
    raw_data = df[feature_cols].values
    n = len(raw_data)
    train_end = int(n * train_ratio)

    feature_scaler = MinMaxScaler(feature_range=(0, 1))
    feature_scaler.fit(raw_data[:train_end])
    scaled_data = feature_scaler.transform(raw_data)

    target_scaler = MinMaxScaler(feature_range=(0, 1))
    target_scaler.min_ = np.array([feature_scaler.min_[target_idx]])
    target_scaler.scale_ = np.array([feature_scaler.scale_[target_idx]])

    all_X, all_y = create_sliding_sequences(scaled_data, target_idx, seq_length, forecast_horizon)
    num_samples = len(all_X)
    if num_samples < 5:
        raise ValueError(f'Not enough sequences ({num_samples}) with seq_length={seq_length}, horizon={forecast_horizon}.')

    train_count = int(num_samples * train_ratio)
    val_count = max(1, int(num_samples * val_ratio))
    if train_count + val_count >= num_samples:
        train_count = max(1, num_samples - val_count - 1)
    test_start = train_count + val_count

    X_train, y_train = all_X[:train_count], all_y[:train_count]
    X_val, y_val = all_X[train_count:test_start], all_y[train_count:test_start]
    X_test, y_test = all_X[test_start:], all_y[test_start:]

    all_dates = df.index[seq_length : seq_length + num_samples]
    test_dates = all_dates[test_start:]

    return {
        'X_train': X_train, 'y_train': y_train,
        'X_val': X_val, 'y_val': y_val,
        'X_test': X_test, 'y_test': y_test,
        'feature_scaler': feature_scaler, 'target_scaler': target_scaler,
        'feature_cols': feature_cols, 'target_col': target_col,
        'target_idx': target_idx, 'seq_length': seq_length,
        'forecast_horizon': forecast_horizon, 'test_dates': test_dates,
    }

print('[+] Preprocessing engine loaded.')


---
## Step 4: Training Engine & Metric Evaluation

In [ ]:
def format_price(v: float) -> str:
    if abs(v) < 0.01: return f'${v:.5f}'
    elif abs(v) < 5.0: return f'${v:.4f}'
    elif abs(v) < 1000.0: return f'${v:.2f}'
    else: return f'${v:,.1f}'


def train_lstm_model(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader, epochs: int = 30, lr: float = 0.001, device: str = 'cpu', patience: int = 10) -> Dict[str, List[float]]:
    model.to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=4)

    history = {'train_loss': [], 'val_loss': []}
    best_val_loss = float('inf')
    patience_counter = 0
    best_weights = None

    for epoch in range(1, epochs + 1):
        model.train()
        running_train = 0.0
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            preds = model(X_b)
            loss = criterion(preds, y_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            running_train += loss.item() * len(X_b)

        train_loss = running_train / len(train_loader.dataset)

        model.eval()
        running_val = 0.0
        with torch.no_grad():
            for X_b, y_b in val_loader:
                X_b, y_b = X_b.to(device), y_b.to(device)
                preds = model(X_b)
                loss = criterion(preds, y_b)
                running_val += loss.item() * len(X_b)

        val_loss = running_val / len(val_loader.dataset)
        scheduler.step(val_loss)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    if best_weights is not None:
        model.load_state_dict(best_weights)
    return history


def evaluate_predictions(model: nn.Module, X_test: np.ndarray, y_test: np.ndarray, target_scaler: Any, device: str = 'cpu') -> Dict[str, Any]:
    model.eval()
    with torch.no_grad():
        X_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
        scaled_preds = model(X_tensor).cpu().numpy()

    horizon = y_test.shape[1] if y_test.ndim > 1 else 1
    actuals = target_scaler.inverse_transform(y_test.reshape(-1, 1)).reshape(-1, horizon)
    predictions = target_scaler.inverse_transform(scaled_preds.reshape(-1, 1)).reshape(-1, horizon)

    mae = np.mean(np.abs(actuals - predictions))
    rmse = np.sqrt(np.mean((actuals - predictions) ** 2))
    mape = np.mean(np.abs((actuals - predictions) / (actuals + 1e-9))) * 100
    step_rmse = [float(np.sqrt(np.mean((actuals[:, s] - predictions[:, s]) ** 2))) for s in range(horizon)]

    if horizon == 1:
        actual_dir = np.diff(actuals.flatten()) > 0
        pred_dir = np.diff(predictions.flatten()) > 0
    else:
        actual_dir = (actuals[:, -1] - actuals[:, 0]) > 0
        pred_dir = (predictions[:, -1] - predictions[:, 0]) > 0
    dir_acc = np.mean(actual_dir == pred_dir) * 100 if len(actual_dir) > 0 else 0.0

    return {
        'actuals': actuals, 'predictions': predictions,
        'mae': float(mae), 'rmse': float(rmse), 'mape': float(mape),
        'step_rmse': step_rmse, 'directional_accuracy': float(dir_acc),
        'forecast_horizon': horizon,
    }

print('[+] Training & evaluation routines ready.')


---
## Step 5: DATA INGESTION (Upload Custom Dataset / File / API / Public Ticker)

Choose your data source below:
1. **Upload CSV / File**: Upload a file from your computer directly into Colab.
2. **Local / Drive Path**: Read from a path like `/content/my_data.csv` or Google Drive.
3. **Custom REST API**: Connect your own JSON API endpoint.
4. **Public Ticker**: Fetch live market data (e.g. `BTC-USD`, `AAPL`, `TSLA`).


In [ ]:
#@title Select Data Source & Settings { run: "auto" }
DATA_SOURCE = "Upload CSV / File" #@param ["Upload CSV / File", "Local / Drive File Path", "Custom REST API", "Public Ticker"]
CUSTOM_FILE_PATH = "/content/my_data.csv" #@param {type:"string"}

# Public ticker or REST API fallback settings:
SYMBOL = "BTC-USD" #@param {type:"string"}
INTERVAL = "1h" #@param ["1m", "5m", "15m", "1h", "1d"]
PERIOD = "60d" #@param ["5d", "30d", "60d", "1y", "2y"]
MY_API_URL = "" #@param {type:"string"}
MY_API_KEY = "" #@param {type:"string"}

# Training hyperparameters:
FORECAST_HORIZON = 5 #@param {type:"slider", min:1, max:15, step:1}
SEQ_LENGTH = 24 #@param {type:"slider", min:10, max:60, step:2}
EPOCHS = 15 #@param {type:"slider", min:5, max:50, step:5}


def standardize_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = [str(c).strip() for c in df.columns]
    date_candidates = ['date', 'datetime', 'timestamp', 'time', 'date_time', 'dt']
    matched = None
    for col in df.columns:
        if col.lower() in date_candidates:
            matched = col
            break
    if matched:
        df['Datetime'] = pd.to_datetime(df[matched], errors='coerce')
        df = df.set_index('Datetime')
        if matched != 'Datetime' and matched in df.columns:
            df = df.drop(columns=[matched])
    elif not isinstance(df.index, pd.DatetimeIndex):
        try:
            df.index = pd.to_datetime(df.index)
        except Exception:
            df.index = pd.date_range(start='2024-01-01', periods=len(df), freq='D')

    col_map = {}
    for col in df.columns:
        cl = col.lower()
        if cl in ['open', 'o']: col_map[col] = 'Open'
        elif cl in ['high', 'h']: col_map[col] = 'High'
        elif cl in ['low', 'l']: col_map[col] = 'Low'
        elif cl in ['close', 'c', 'price', 'last', 'settle', 'adj close', 'adj_close']:
            if 'Close' not in col_map.values(): col_map[col] = 'Close'
        elif cl in ['volume', 'vol', 'v', 'qty', 'quantity']: col_map[col] = 'Volume'
    df = df.rename(columns=col_map)

    if 'Close' not in df.columns:
        num_cols = df.select_dtypes(include=[np.number]).columns
        if len(num_cols) > 0:
            df['Close'] = df[num_cols[0]]
        else:
            raise ValueError(f'No price column found in: {list(df.columns)}')

    if 'Open' not in df.columns: df['Open'] = df['Close']
    if 'High' not in df.columns: df['High'] = df['Close']
    if 'Low' not in df.columns: df['Low'] = df['Close']
    if 'Volume' not in df.columns: df['Volume'] = 1.0

    for c in ['Open', 'High', 'Low', 'Close', 'Volume']:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    return df.sort_index().dropna(subset=['Close'])


df_raw = None
dataset_label = SYMBOL

if DATA_SOURCE == "Upload CSV / File":
    try:
        from google.colab import files
        print('[*] Click the "Choose Files" button below to upload your dataset (CSV, Excel, etc.):')
        uploaded = files.upload()
        if uploaded:
            uploaded_filename = list(uploaded.keys())[0]
            dataset_label = uploaded_filename
            if uploaded_filename.endswith(('.xlsx', '.xls')):
                df_raw = pd.read_excel(uploaded_filename)
            else:
                df_raw = pd.read_csv(uploaded_filename)
            df_raw = standardize_dataframe(df_raw)
            print(f'[+] Successfully parsed uploaded file: {uploaded_filename}')
        else:
            print('[!] No file uploaded. Falling back to public ticker feed.')
    except ImportError:
        print('[!] Not running in Colab or file upload cancelled. Checking for local file...')

if df_raw is None and DATA_SOURCE == "Local / Drive File Path":
    if os.path.exists(CUSTOM_FILE_PATH):
        dataset_label = os.path.basename(CUSTOM_FILE_PATH)
        df_raw = pd.read_excel(CUSTOM_FILE_PATH) if CUSTOM_FILE_PATH.endswith(('.xlsx', '.xls')) else pd.read_csv(CUSTOM_FILE_PATH)
        df_raw = standardize_dataframe(df_raw)
        print(f'[+] Loaded dataset from path: {CUSTOM_FILE_PATH}')
    else:
        print(f'[!] File not found at {CUSTOM_FILE_PATH}. Falling back to public ticker.')

if df_raw is None and DATA_SOURCE == "Custom REST API" and MY_API_URL.strip():
    print(f'[*] Ingesting from REST API: {MY_API_URL}...')
    headers = {'Authorization': f'Bearer {MY_API_KEY}'} if MY_API_KEY else {}
    resp = requests.get(MY_API_URL, headers=headers, params={'symbol': SYMBOL, 'limit': 1000}, timeout=15)
    resp.raise_for_status()
    df_raw = standardize_dataframe(pd.DataFrame(resp.json()))

if df_raw is None:
    print(f'[*] Fetching public feed for {SYMBOL} ({INTERVAL}, {PERIOD})...')
    raw = yf.download(SYMBOL, period=PERIOD, interval=INTERVAL, progress=False)
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)
    df_raw = standardize_dataframe(raw.dropna())

df = add_technical_indicators(df_raw)
print(f'[+] Ready! {len(df)} candles with indicators from {df.index[0]} to {df.index[-1]}.')
display(df.tail(3))


---
## Step 6: Train Phase 1 & Benchmark All Feature Sets
Trains and compares 4 feature architectures on your GPU.

In [ ]:
feature_sets = {
    'Univariate (Close Only)': ['Close'],
    'Basic Market (OHLCV)': ['Open', 'High', 'Low', 'Close', 'Volume'],
    'Market + Trend (MAs)': ['Open', 'High', 'Low', 'Close', 'Volume', 'SMA_10', 'SMA_20', 'EMA_12'],
    'Full Multi-Feature': ['Open', 'High', 'Low', 'Close', 'Volume', 'SMA_20', 'RSI_14', 'MACD', 'MACD_Signal', 'BB_Width', 'Volatility_20'],
}

results = {}
os.makedirs('./colab_outputs', exist_ok=True)

for name, features in feature_sets.items():
    print(f'[*] Training: {name} ({len(features)} features) | Horizon: {FORECAST_HORIZON} candles...')
    prep = prepare_datasets(df, feature_cols=features, target_col='Close', seq_length=SEQ_LENGTH, forecast_horizon=FORECAST_HORIZON)

    train_loader = DataLoader(StockDataset(prep['X_train'], prep['y_train']), batch_size=32, shuffle=True)
    val_loader = DataLoader(StockDataset(prep['X_val'], prep['y_val']), batch_size=32, shuffle=False)

    model = StockLSTM(input_size=len(features), hidden_size=64, num_layers=2, output_size=FORECAST_HORIZON, dropout=0.2)
    history = train_lstm_model(model, train_loader, val_loader, epochs=EPOCHS, device=device.type)
    eval_res = evaluate_predictions(model, prep['X_test'], prep['y_test'], prep['target_scaler'], device=device.type)

    ckpt_path = f'./colab_outputs/lstm_{name.replace(" ", "_").lower()}.pt'
    torch.save({
        'model_state': model.state_dict(), 'feature_cols': features,
        'feature_scaler': prep['feature_scaler'], 'target_scaler': prep['target_scaler'],
        'seq_length': SEQ_LENGTH, 'forecast_horizon': FORECAST_HORIZON, 'hidden_size': 64,
    }, ckpt_path)

    results[name] = {'model': model, 'history': history, 'metrics': eval_res, 'prep': prep, 'ckpt_path': ckpt_path}
    print(f'    -> RMSE: {format_price(eval_res["rmse"])} | MAE: {format_price(eval_res["mae"])} | Dir Acc: {eval_res["directional_accuracy"]:.1f}%')

# Identify Best Model
best_set_name = min(results.keys(), key=lambda k: results[k]['metrics']['rmse'])
best_ckpt = results[best_set_name]['ckpt_path']
print('\n' + '='*65)
print(f'[BEST] Top Feature Set: {best_set_name} (RMSE: {format_price(results[best_set_name]["metrics"]["rmse"])})')
print(f'       Saved Checkpoint: {best_ckpt}')
print('='*65)


---
## Step 7: Graphical Visualizations (Multi-Step Trajectories & Loss Curves)

In [ ]:
# 1. Model Predictions vs Ground Truth Overlay
plt.figure(figsize=(15, 6))
first_key = list(results.keys())[0]
test_dates = results[first_key]['prep']['test_dates']
actuals_step1 = results[first_key]['metrics']['actuals'][:, 0]
plt.plot(test_dates, actuals_step1, label='Ground Truth (Actual Price)', color='black', linewidth=2.5)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
for i, (name, data) in enumerate(results.items()):
    preds_step1 = data['metrics']['predictions'][:, 0]
    rmse = data['metrics']['step_rmse'][0]
    plt.plot(test_dates, preds_step1, label=f'{name} (Step 1 RMSE: {format_price(rmse)})', color=colors[i % len(colors)], linestyle='--', linewidth=1.8)

plt.title(f'{SYMBOL} Model Predictions Comparison on Test Data (Horizon N={FORECAST_HORIZON})', fontsize=13, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend(loc='upper left')
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

# 2. Multi-Step Degradation Curve & Forecast Cones
if FORECAST_HORIZON > 1:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    steps = list(range(1, FORECAST_HORIZON + 1))
    for i, (name, data) in enumerate(results.items()):
        axes[0].plot(steps, data['metrics']['step_rmse'], marker='o', label=name, color=colors[i % len(colors)], linewidth=2)
    axes[0].set_title('RMSE Error Degradation Across Horizon (t+1 to t+N)', fontweight='bold')
    axes[0].set_xlabel('Horizon Step Ahead (t + k)')
    axes[0].set_ylabel('RMSE')
    axes[0].set_xticks(steps)
    axes[0].grid(True, linestyle=':', alpha=0.6)
    axes[0].legend()

    sample_idx = len(test_dates) - 1
    axes[1].plot(steps, results[first_key]['metrics']['actuals'][sample_idx], marker='s', label='Actual Path', color='black', linewidth=2.5)
    for i, (name, data) in enumerate(results.items()):
        axes[1].plot(steps, data['metrics']['predictions'][sample_idx], marker='^', linestyle='--', label=f'{name}', color=colors[i % len(colors)], linewidth=2)
    axes[1].set_title('Latest Window Multi-Step Trajectory Forecast', fontweight='bold')
    axes[1].set_xlabel('Forecast Step (t + k)')
    axes[1].set_ylabel('Price')
    axes[1].set_xticks(steps)
    axes[1].grid(True, linestyle=':', alpha=0.6)
    axes[1].legend()
    plt.tight_layout()
    plt.show()


---
## Step 8: Phase 2 Real-Time Live Inference Engine
Takes the winning Phase 1 checkpoint and performs real-time next-candle trajectory forecasting on live streaming data.

In [ ]:
class ColabLivePredictor:
    def __init__(self, checkpoint_path: str, device: str = 'cpu'):
        ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
        self.feature_cols = ckpt['feature_cols']
        self.feature_scaler = ckpt['feature_scaler']
        self.target_scaler = ckpt['target_scaler']
        self.seq_length = ckpt['seq_length']
        self.forecast_horizon = ckpt.get('forecast_horizon', 1)
        self.device = device

        self.model = StockLSTM(input_size=len(self.feature_cols), hidden_size=64, num_layers=2, output_size=self.forecast_horizon, dropout=0.0)
        self.model.load_state_dict(ckpt['model_state'])
        self.model.to(device)
        self.model.eval()

    def predict(self, live_df: pd.DataFrame) -> Dict[str, Any]:
        processed = add_technical_indicators(live_df) if any('SMA' in f or 'RSI' in f for f in self.feature_cols) else live_df
        recent = processed[self.feature_cols].iloc[-self.seq_length :].values
        scaled = self.feature_scaler.transform(recent)
        tensor_in = torch.tensor(scaled, dtype=torch.float32).unsqueeze(0).to(self.device)

        with torch.no_grad():
            pred_scaled = self.model(tensor_in).cpu().numpy()
        preds = self.target_scaler.inverse_transform(pred_scaled.reshape(-1, 1)).flatten().tolist()
        current_p = float(live_df['Close'].iloc[-1])

        traj = []
        for idx, p in enumerate(preds, start=1):
            d = p - current_p
            pct = (d / current_p) * 100
            traj.append({'step': idx, 'price': round(p, 2), 'delta': round(d, 2), 'pct': round(pct, 2), 'signal': 'BULLISH' if d > 0 else 'BEARISH'})
        return {'timestamp': str(live_df.index[-1]), 'current_price': round(current_p, 2), 'trajectory': traj}


# Run Live Forecast
predictor = ColabLivePredictor(best_ckpt, device=device.type)
forecast = predictor.predict(df_raw)

print('\n' + '#' * 65)
print(f'          LIVE MULTI-STEP TRAJECTORY FORECAST FOR {SYMBOL}')
print('#' * 65)
print(f'  Reference Timestamp:  {forecast["timestamp"]}')
print(f'  Current Live Price:   ${forecast["current_price"]}')
print('\n  Forecasted Horizon Trajectory:')
print(f'  {"Step":<6} | {"Projected Price":<18} | {"Delta vs Current":<20} | {"Signal"}')
print('  ' + '-' * 60)
for t in forecast['trajectory']:
    print(f'  t+{t["step"]:.<3} | ${t["price"]:.<17} | {t["delta"]:>+10.2f} ({t["pct"]:>+5.2f}%)   | {t["signal"]}')
print('  ' + '-' * 60)
terminal = forecast['trajectory'][-1]
print(f'  Net Terminal Target (t+{len(forecast["trajectory"])}): ${terminal["price"]} ({terminal["delta"]:+.2f} | {terminal["pct"]:+.2f}%) -> {terminal["signal"]}')
print('#' * 65)


---
## Step 9: Save Models Directly to Google Drive (Optional)
Uncomment and run below to save all trained `.pt` checkpoints directly to your Google Drive.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/Stock_LSTM_Models
# !cp -r ./colab_outputs/* /content/drive/MyDrive/Stock_LSTM_Models/
# print('[+] Saved trained checkpoints to Google Drive!')
